In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q ultralytics

In [ ]:
import os
import glob
import random
import shutil
import yaml
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from PIL import Image
from tqdm.auto import tqdm

import torch
from ultralytics import YOLO

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

In [ ]:
from pathlib import Path
import os

DATASET_DIR = Path("/kaggle/input/datasets")

print("Dataset location:")
print(DATASET_DIR)

print("\nDataset contents:")
print("=" * 60)

for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(str(DATASET_DIR), "").count(os.sep)

    if level > 3:
        continue

    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:20]:
        print(f"{indent}    {file}")

In [ ]:
from pathlib import Path

DATASET_ROOT = Path("/kaggle/input/datasets/gbiamgaurav/racoon-object-detection")
DATA_YAML = DATASET_ROOT / "data.yaml"

print("Dataset root:")
print(DATASET_ROOT)

print("\nContents:")
for item in DATASET_ROOT.iterdir():
    print(item)

print("\n" + "=" * 60)
print("data.yaml")
print("=" * 60)

with open(DATA_YAML, "r") as f:
    print(f.read())

In [ ]:
from pathlib import Path

for split in ["train", "valid", "test"]:
    split_dir = DATASET_ROOT / split

    print("\n" + "=" * 60)
    print(f"{split.upper()}")
    print("=" * 60)

    if not split_dir.exists():
        print("Directory not found!")
        continue

    for item in split_dir.iterdir():
        print(item.name)

    # Count images and labels recursively
    images = list(split_dir.rglob("*.jpg")) + \
             list(split_dir.rglob("*.jpeg")) + \
             list(split_dir.rglob("*.png"))

    labels = list(split_dir.rglob("*.txt"))

    print(f"\nImages : {len(images)}")
    print(f"Labels : {len(labels)}")

In [ ]:
import ultralytics

print("Ultralytics version:", ultralytics.__version__)

In [ ]:
from pathlib import Path
import yaml

DATASET_ROOT = Path("/kaggle/input/datasets/gbiamgaurav/racoon-object-detection")
DATA_YAML = DATASET_ROOT / "data.yaml"

with open(DATA_YAML, "r") as f:
    data = yaml.safe_load(f)

print("DATASET CONFIGURATION")
print("=" * 60)

for key, value in data.items():
    print(f"{key}: {value}")

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

TRAIN_IMAGES = DATASET_ROOT / "train" / "images"
TRAIN_LABELS = DATASET_ROOT / "train" / "labels"

image_files = list(TRAIN_IMAGES.glob("*"))

random.seed(42)
selected_images = random.sample(image_files, min(6, len(image_files)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, image_path in zip(axes, selected_images):

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w = image.shape[:2]

    label_path = TRAIN_LABELS / f"{image_path.stem}.txt"

    if label_path.exists():

        with open(label_path, "r") as f:
            lines = f.readlines()

        for line in lines:
            values = line.strip().split()

            if len(values) != 5:
                continue

            class_id, x_center, y_center, box_width, box_height = map(
                float, values
            )

            x_center *= w
            y_center *= h
            box_width *= w
            box_height *= h

            x1 = int(x_center - box_width / 2)
            y1 = int(y_center - box_height / 2)
            x2 = int(x_center + box_width / 2)
            y2 = int(y_center + box_height / 2)

            cv2.rectangle(
                image,
                (x1, y1),
                (x2, y2),
                (255, 0, 0),
                3
            )

            cv2.putText(
                image,
                "raccoon",
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 0, 0),
                2
            )

    ax.imshow(image)
    ax.set_title(image_path.name)
    ax.axis("off")

for ax in axes[len(selected_images):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from ultralytics import YOLO


model = YOLO("yolov8n.pt")

print("Pretrained YOLO model loaded successfully.")

In [ ]:
results = model.train(
    data=str(DATA_YAML),

    epochs=100,
    imgsz=640,
    batch=16,

    pretrained=True,

    optimizer="auto",

    dropout=0.10,

    patience=15,

    seed=42,

    device=0,

    project="/kaggle/working/raccoon_detection",
    name="yolov8s_raccoon",

    plots=True,

    verbose=True
)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

BEST_MODEL = "/kaggle/working/raccoon_detection/yolov8s_raccoon/weights/best.pt"

best_model = YOLO(BEST_MODEL)

print("Best model loaded successfully.")
print("Model:", BEST_MODEL)

In [ ]:
test_results = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    verbose=True
)

In [ ]:
print("=" * 60)
print("TEST SET PERFORMANCE")
print("=" * 60)

print(f"Overall Precision : {test_results.box.mp:.4f}")
print(f"Overall Recall    : {test_results.box.mr:.4f}")
print(f"Overall mAP@50    : {test_results.box.map50:.4f}")
print(f"Overall mAP@50:95 : {test_results.box.map:.4f}")

In [ ]:
class_names = ['Cat', 'Dog', 'Human', 'Possum', 'racoon']

print("\nPer-class TEST results:")
print("-" * 75)

for i, name in enumerate(class_names):
    if i < len(test_results.box.p):
        print(
            f"{name:10s} | "
            f"Precision: {test_results.box.p[i]:.4f} | "
            f"Recall: {test_results.box.r[i]:.4f} | "
            f"mAP50: {test_results.box.ap50[i]:.4f} | "
            f"mAP50-95: {test_results.box.ap[i]:.4f}"
        )

In [ ]:
from pathlib import Path
from IPython.display import Image, display

RUN_DIR = Path("/kaggle/working/raccoon_detection/yolov8s_raccoon")

confusion_matrix = RUN_DIR / "confusion_matrix.png"

if confusion_matrix.exists():
    display(Image(filename=str(confusion_matrix)))
else:
    print("Confusion matrix not found in:", RUN_DIR)

In [ ]:
results_plot = RUN_DIR / "results.png"

if results_plot.exists():
    display(Image(filename=str(results_plot)))
else:
    print("Training results plot not found.")

In [ ]:
TEST_IMAGES = DATASET_ROOT / "test" / "images"

prediction_results = best_model.predict(
    source=str(TEST_IMAGES),
    imgsz=640,
    conf=0.25,
    device=0,
    save=True,
    project="/kaggle/working/raccoon_detection",
    name="test_predictions",
    verbose=False
)

print("Predictions generated.")
print("Saved to:")
print("/kaggle/working/raccoon_detection/test_predictions")

In [ ]:
import glob
from IPython.display import Image, display

prediction_files = glob.glob(
    "/kaggle/working/raccoon_detection/test_predictions/*"
)

print("Number of prediction images:", len(prediction_files))

for file in prediction_files[:6]:
    print(file)
    display(Image(filename=file))

In [ ]:
total_detections = 0
raccoon_detections = 0

print("=" * 70)
print("TEST PREDICTION SUMMARY")
print("=" * 70)

for result in prediction_results:

    image_name = Path(result.path).name

    if result.boxes is None:
        print(f"{image_name}: No detections")
        continue

    classes = result.boxes.cls.cpu().numpy()
    confidences = result.boxes.conf.cpu().numpy()

    total = len(classes)
    total_detections += total

    names = [best_model.names[int(c)] for c in classes]

    raccoon_count = sum(
        1 for name in names
        if name.lower() == "racoon"
    )

    raccoon_detections += raccoon_count

    print(
        f"{image_name:45s} "
        f"Detections: {total:2d} | "
        f"Racoon: {raccoon_count:2d}"
    )

print("\n" + "=" * 70)
print(f"Total detections       : {total_detections}")
print(f"Total racoon detections: {raccoon_detections}")

In [ ]:
import shutil
from pathlib import Path

source_model = Path(
    "/kaggle/working/raccoon_detection/"
    "yolov8s_raccoon/weights/best.pt"
)

destination_model = Path(
    "/kaggle/working/raccoon_yolov8_best.pt"
)

shutil.copy2(source_model, destination_model)

print("Final model saved:")
print(destination_model)
print(f"Size: {destination_model.stat().st_size / (1024**2):.2f} MB")